# Self-Healing Machine Learning (SHML)

Automated Failure Detection, Diagnosis, Recovery,
and Safety Validation for Credit-Default Prediction


# Phase C — Controlled Missingness and Recovery

## Research Objective

Evaluate whether SHML's existing median-imputation
strategy maintains or improves predictive performance
compared with a predefined zero-imputation benchmark.

## Research Hypotheses

H1: Increasing BILL_AMT1 missingness produces measurable
predictive degradation under at least some conditions.

H2: Recovery effectiveness differs between imputation
strategies.

H4: The magnitude of performance changes depends on
missingness severity.

## Experimental Configuration

- Dataset: UCI Default of Credit Card Clients
- Model: Existing Logistic Regression baseline
- Corrupted feature: BILL_AMT1 only
- Missingness: 0%, 5%, 15%, 30%, 60%
- Random seed: 42 for the initial experiment
- Classification threshold: 0.50
- Healthy reference: Uncorrupted validation data
- Benchmark: Zero imputation
- SHML recovery: Training-median imputation
- Evaluation: F1, precision, recall, accuracy, ROC-AUC

## Experimental Controls

The model, training data, validation observations,
random missingness masks, and classification threshold
remain fixed between recovery strategies.

The test dataset remains untouched.

This initial experiment is exploratory. Statistical
validation across additional seeds follows in Phase D.


In [1]:

import numpy as np


def inject_bill_amt1_missingness(
    X_batch,
    missingness_rate,
    seed=42
):
    """
    Inject MCAR missingness into BILL_AMT1 only.

    Returns:
        corrupted DataFrame
        Boolean mask of modified rows
    """

    if not 0 <= missingness_rate <= 1:
        raise ValueError(
            "Missingness rate must be between 0 and 1."
        )

    if "BILL_AMT1" not in X_batch.columns:
        raise ValueError("BILL_AMT1 is missing.")

    X_corrupted = X_batch.copy(deep=True)

    rng = np.random.default_rng(seed)

    n_missing = int(
        len(X_corrupted) * missingness_rate
    )

    selected_positions = rng.choice(
        len(X_corrupted),
        size=n_missing,
        replace=False
    )

    mask = np.zeros(
        len(X_corrupted),
        dtype=bool
    )

    mask[selected_positions] = True

    X_corrupted.iloc[
        selected_positions,
        X_corrupted.columns.get_loc("BILL_AMT1")
    ] = np.nan

    return X_corrupted, mask


In [4]:
import pandas as pd

# Lightweight function test.
# No model training or dataset loading needed.
sample = pd.DataFrame({
    "BILL_AMT1": range(100),
    "BILL_AMT2": range(100, 200)
})

corrupted, mask = inject_bill_amt1_missingness(
    sample,
    missingness_rate=0.15,
    seed=42
)

assert corrupted["BILL_AMT1"].isna().sum() == 15
assert corrupted["BILL_AMT2"].equals(sample["BILL_AMT2"])
assert sample.isna().sum().sum() == 0
assert mask.sum() == 15

print("Phase C missingness injection test passed!")

Phase C missingness injection test passed!


In [5]:

from src.detection import detect_feature_missingness
from src.diagnosis import diagnose_failure
from src.recovery import recover_missingness
from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety


def compare_recovery_strategies(
    X_val,
    y_val,
    X_train,
    baseline_model,
    scaler,
    missingness_rate,
    seed=42
):
    """
    Compare a zero-fill benchmark with SHML median recovery.

    Reuses the existing trained model and SHML modules.
    """

    # 1. Inject missingness
    X_corrupted, mask = inject_bill_amt1_missingness(
        X_val,
        missingness_rate,
        seed
    )

    # 2. Healthy reference
    healthy_metrics = evaluate_model(
        X_batch=X_val,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    # 3. Zero-imputation benchmark
    X_zero = X_corrupted.copy(deep=True)
    X_zero["BILL_AMT1"] = (
        X_zero["BILL_AMT1"].fillna(0)
    )

    zero_metrics = evaluate_model(
        X_batch=X_zero,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    # 4. Existing SHML detection
    detection = detect_feature_missingness(
        X_batch=X_corrupted,
        feature="BILL_AMT1",
        reference_missingness=0.0,
        threshold=0.05
    )

    # 5. Existing SHML diagnosis
    diagnosis = diagnose_failure(detection)

    # 6. Existing SHML recovery
    training_median = X_train["BILL_AMT1"].median()

    X_recovered, recovery_log = recover_missingness(
        X_batch=X_corrupted,
        diagnosis=diagnosis,
        training_median=training_median
    )

    # 7. Evaluate SHML recovery
    recovered_metrics = evaluate_model(
        X_batch=X_recovered,
        y_true=y_val,
        model=baseline_model,
        scaler=scaler
    )

    # 8. Verify recovery integrity
    integrity_passed = (
        X_recovered.isna().sum().sum() == 0
        and X_recovered.shape == X_corrupted.shape
        and X_recovered.columns.equals(X_corrupted.columns)
        and X_recovered.loc[
            ~mask, "BILL_AMT1"
        ].equals(
            X_corrupted.loc[~mask, "BILL_AMT1"]
        )
        and X_recovered.drop(
            columns=["BILL_AMT1"]
        ).equals(
            X_corrupted.drop(columns=["BILL_AMT1"])
        )
        and (
            X_recovered.loc[mask, "BILL_AMT1"]
            == training_median
        ).all()
    )

    # 9. Existing SHML safety mechanism
    safety = evaluate_recovery_safety(
        recovered_metrics=recovered_metrics,
        healthy_metrics=healthy_metrics,
        data_integrity_passed=integrity_passed,
        max_f1_loss=0.10,
        max_recall_loss=0.10
    )

    # 10. Collect paired results
    return {
        "Missingness": missingness_rate,
        "Seed": seed,
        "Values_Missing": int(mask.sum()),
        "Detected": detection["failure_detected"],
        "Healthy_F1": healthy_metrics["f1"],
        "Zero_F1": zero_metrics["f1"],
        "SHML_F1": recovered_metrics["f1"],
        "Zero_Recall": zero_metrics["recall"],
        "SHML_Recall": recovered_metrics["recall"],
        "Zero_ROC_AUC": zero_metrics["roc_auc"],
        "SHML_ROC_AUC": recovered_metrics["roc_auc"],
        "F1_Improvement": (
            recovered_metrics["f1"] -
            zero_metrics["f1"]
        ),
        "Integrity_Passed": integrity_passed,
        "Safety_Decision": safety["decision"],
        "Safety_Reason": safety["reason"]
    }


In [8]:
import pandas as pd

missingness_levels = [0.00, 0.05, 0.15, 0.30, 0.60]
experiment_results = []

for rate in missingness_levels:
    result = compare_recovery_strategies(
        X_val=X_val,
        y_val=y_val,
        X_train=X_train,
        baseline_model=baseline_model,
        scaler=scaler,
        missingness_rate=rate,
        seed=42
    )
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results)
display(results_df.round(6))

NameError: name 'X_val' is not defined